# 🐝 말벌 AI 다시 가르치기 (VespAI 파인튜닝)

## 사용법 — 코드는 안 고쳐도 돼요
1. 위 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU → 저장**
2. 위 메뉴 **런타임 → 모두 실행**
3. "Google Drive에 연결" 창이 뜨면 → **계정 선택 → 허용**
4. 끝날 때까지 기다리기 (30분~1시간). **창을 닫으면 멈춰요.**
5. 끝나면 구글 드라이브 `hornet_finetune` 폴더에 **`best_hornet_v1.pt`** 가 생겨요. 이게 새로 공부한 AI예요.

❗ 빨간 에러가 나면 그 부분을 캡처해서 Claude에게 보내 주세요.

### 실행 전에 준비할 것
- 구글 드라이브에 `hornet_finetune` 폴더 만들기
- 그 안에 Roboflow에서 받은 zip 파일 넣기 (이름은 안 바꿔도 돼요)
- Roboflow 라벨 이름은 소문자로 정확히: 등검은말벌 = `velutina`, 비슷한 다른 곤충 = `other`

## 1. GPU 확인 (표가 나오면 정상)

In [ ]:
!nvidia-smi

## 2. 설정 — 그대로 두면 돼요

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/hornet_finetune'   # 데이터와 결과가 있는 드라이브 폴더
os.makedirs(DRIVE_DIR, exist_ok=True)
DATA_ZIP = ''            # 비워 두면 폴더 안의 zip을 자동으로 찾음
CLASSES  = ['other', 'velutina']   # Roboflow zip이면 자동으로 바뀜
VESPAI_WEIGHTS = ''      # 비워 두면 VespAI 가중치를 자동으로 받음
IMG_SIZE = 640
EPOCHS   = 100
BATCH    = 16            # 'out of memory' 에러가 나면 8로
FREEZE   = 10            # 사진이 1500장보다 많아지면 0으로 바꿔 보기
RUN_NAME = 'hornet_v1'
SEED     = 0

## 3. YOLOv5 설치

In [ ]:
%cd /content
!test -d yolov5 || git clone -q https://github.com/ultralytics/yolov5
%cd /content/yolov5
!pip install -q -r requirements.txt
# 최신 버전끼리 안 맞는 그림 저장 부분 고치기 (빨간 에러 방지)
!sed -i 's/annotator.im.save(fname)/Image.fromarray(np.asarray(annotator.im)).save(fname)/' utils/plots.py

## 4. VespAI 가중치 받기 (VespAI가 지금까지 공부한 내용)

In [ ]:
import glob, subprocess, torch
import sys; sys.path.insert(0, '/content/yolov5')
PREFERRED = 'yolov5s-all-data.pt'   # VespAI가 모든 데이터로 학습한 파일
if not VESPAI_WEIGHTS:
    if not os.path.exists('/content/vespai'):
        subprocess.run(['git', 'clone', '-q', '--depth', '1',
                        'https://github.com/andrw3000/vespai', '/content/vespai'])
    pts = [p for p in sorted(glob.glob('/content/vespai/models/yolov5-params/*.pt'))
           if os.path.getsize(p) > 5e6]
    pick = [p for p in pts if p.endswith(PREFERRED)] or pts
    VESPAI_WEIGHTS = pick[0] if pick else 'yolov5s.pt'
print('시작 가중치:', VESPAI_WEIGHTS)
try:
    ck = torch.load(VESPAI_WEIGHTS, map_location='cpu', weights_only=False)
    print('VespAI가 원래 알던 것:', ck['model'].names)
except Exception as e:
    print('(확인 생략)', e)

## 5. 사진 정리 (zip 풀고 학습용 / 검사용 / 시험용으로 나누기)

In [ ]:
import zipfile, shutil, random, pathlib, yaml
from collections import Counter

zip_path = DATA_ZIP
if not zip_path:
    zips = sorted(glob.glob(f'{DRIVE_DIR}/*.zip'), key=os.path.getmtime)
    if not zips:
        raise FileNotFoundError('❌ 구글 드라이브 hornet_finetune 폴더에 zip 파일이 없어요. '
                                'Roboflow에서 받은 zip을 넣고 다시 [런타임 → 모두 실행] 하세요.')
    zip_path = zips[-1]
print('데이터 파일:', zip_path)

RAW = pathlib.Path('/content/raw')
DS  = pathlib.Path('/content/dataset')
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp'}
shutil.rmtree(RAW, ignore_errors=True); shutil.rmtree(DS, ignore_errors=True)
zipfile.ZipFile(zip_path).extractall(RAW)

rf_yaml = list(RAW.rglob('data.yaml'))
if rf_yaml:
    # Roboflow 형식 (YOLOv5 PyTorch)
    root = rf_yaml[0].parent
    y = yaml.safe_load(open(rf_yaml[0]))
    names = y['names']
    CLASSES = [names[k] for k in sorted(names)] if isinstance(names, dict) else list(names)
    data = {'path': str(root), 'train': 'train/images', 'val': 'valid/images', 'names': CLASSES}
    if (root / 'test/images').exists():
        data['test'] = 'test/images'
    print('Roboflow 형식 확인 ✅')
else:
    # LabelImg / CVAT 형식: 사진 + 같은 이름 .txt  (파일 이름 맨 앞 '_' 전까지를 표본 번호로 보고 나눔)
    imgs = sorted(p for p in RAW.rglob('*') if p.suffix.lower() in IMG_EXT)
    labels = {p.stem: p for p in RAW.rglob('*.txt') if p.name != 'classes.txt'}
    groups = {}
    for p in imgs:
        groups.setdefault(p.stem.split('_')[0], []).append(p)
    rng = random.Random(SEED)
    split = {}
    if len(groups) >= 3:
        keys = sorted(groups); rng.shuffle(keys)
        n_val = max(1, round(len(keys) * 0.1)); n_test = max(1, round(len(keys) * 0.1))
        for i, k in enumerate(keys):
            s = 'val' if i < n_val else 'test' if i < n_val + n_test else 'train'
            for p in groups[k]: split[p] = s
    else:
        shuffled = imgs[:]; rng.shuffle(shuffled)
        n = len(shuffled)
        for i, p in enumerate(shuffled):
            split[p] = 'val' if i < n * 0.1 else 'test' if i < n * 0.2 else 'train'
    for p, s in split.items():
        (DS / s / 'images').mkdir(parents=True, exist_ok=True)
        (DS / s / 'labels').mkdir(parents=True, exist_ok=True)
        name = f'{p.parent.name}_{p.name}' if (DS / s / 'images' / p.name).exists() else p.name
        shutil.copy(p, DS / s / 'images' / name)
        dst = DS / s / 'labels' / (pathlib.Path(name).stem + '.txt')
        if p.stem in labels: shutil.copy(labels[p.stem], dst)
        else: dst.write_text('')
    data = {'path': str(DS), 'train': 'train/images', 'val': 'val/images',
            'test': 'test/images', 'names': CLASSES}

yaml.safe_dump(data, open('/content/data.yaml', 'w'), allow_unicode=True)
ROOT = pathlib.Path(data['path'])
test_dir = ROOT / data.get('test', data['val'])

print('클래스:', CLASSES)
for s in ['train', 'val', 'test']:
    if s not in data: continue
    img_dir = ROOT / data[s]; lab_dir = img_dir.parent / 'labels'
    ims = [p for p in img_dir.glob('*') if p.suffix.lower() in IMG_EXT]
    cnt, bg = Counter(), 0
    for p in ims:
        t = lab_dir / (p.stem + '.txt')
        lines = t.read_text().strip().splitlines() if t.exists() else []
        if not lines: bg += 1
        for l in lines: cnt[CLASSES[int(l.split()[0])]] += 1
    print(f'{s:5s}: 사진 {len(ims):4d}장 | 빈 화면 {bg:4d}장 | 네모 {dict(cnt)}')

## 6. 네모(라벨)가 제대로 그려졌는지 확인 — 빨간 네모 = 등검은말벌

In [ ]:
import cv2, matplotlib.pyplot as plt
img_dir = ROOT / data['train']
all_imgs = sorted(p for p in img_dir.glob('*') if p.suffix.lower() in IMG_EXT)
samples = random.Random(1).sample(all_imgs, k=min(8, len(all_imgs)))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, p in zip(axes.flat, samples):
    im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]
    t = img_dir.parent / 'labels' / (p.stem + '.txt')
    for l in (t.read_text().strip().splitlines() if t.exists() else []):
        c, x, y, bw, bh = map(float, l.split()[:5])
        name = CLASSES[int(c)]
        color = (255, 0, 0) if name == 'velutina' else (0, 128, 255)
        x1, y1 = int((x - bw/2) * w), int((y - bh/2) * h)
        x2, y2 = int((x + bw/2) * w), int((y + bh/2) * h)
        cv2.rectangle(im, (x1, y1), (x2, y2), color, 3)
        cv2.putText(im, name, (x1, max(y1 - 8, 20)), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    ax.imshow(im); ax.set_title(p.name[:30], fontsize=8); ax.axis('off')
for ax in axes.flat[len(samples):]: ax.axis('off')
plt.tight_layout(); plt.show()

## 7. 공부하기 전: 원래 VespAI는 우리 사진을 어떻게 보는지 (발표용 '전' 사진)

In [ ]:
!python detect.py --weights {VESPAI_WEIGHTS} --source {test_dir} --img {IMG_SIZE} --conf-thres 0.4 \
    --project /content/before --name vespai --exist-ok --max-det 10

## 8. 공부하기 (제일 오래 걸려요)
결과는 드라이브 `hornet_finetune/runs/`에 바로 저장돼요.

In [ ]:
!python train.py --img {IMG_SIZE} --batch {BATCH} --epochs {EPOCHS} \
    --data /content/data.yaml --weights {VESPAI_WEIGHTS} --freeze {FREEZE} \
    --patience 20 --seed {SEED} --cache \
    --project {DRIVE_DIR}/runs --name {RUN_NAME} --exist-ok

## 9. 성적표 보기

In [ ]:
from IPython.display import Image, display
RUN = f'{DRIVE_DIR}/runs/{RUN_NAME}'
BEST = f'{RUN}/weights/best.pt'
for f in ['results.png', 'confusion_matrix.png']:
    if os.path.exists(f'{RUN}/{f}'):
        print(f); display(Image(f'{RUN}/{f}', width=900))
task = 'test' if 'test' in data else 'val'
!python val.py --weights {BEST} --data /content/data.yaml --img {IMG_SIZE} --task {task} \
    --project /content/eval --name {RUN_NAME} --exist-ok

## 10. 공부한 뒤: 시험 사진에 돌려 보기 (발표용 '후' 사진)

In [ ]:
!python detect.py --weights {BEST} --source {test_dir} --img {IMG_SIZE} --conf-thres 0.4 \
    --project /content/after --name {RUN_NAME} --exist-ok --max-det 10
outs = sorted(pathlib.Path(f'/content/after/{RUN_NAME}').glob('*'))[:8]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, p in zip(axes.flat, outs):
    ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)); ax.set_title(p.name[:30], fontsize=8); ax.axis('off')
for ax in axes.flat[len(outs):]: ax.axis('off')
plt.tight_layout(); plt.show()

## 11. 결과 저장

In [ ]:
dst = f'{DRIVE_DIR}/best_{RUN_NAME}.pt'
shutil.copy(BEST, dst)
print('✅ 끝! 구글 드라이브에 저장됨:', dst)

---
### 참고 (나중에 필요할 때)
**연결이 끊겨서 8번이 멈췄다면** → 아래 셀을 실행하면 이어서 공부해요.

In [ ]:
# 이어서 학습하려면 아래 줄 맨 앞의 # 를 지우고 이 셀만 실행
# !python train.py --resume {DRIVE_DIR}/runs/{RUN_NAME}/weights/last.pt

**라즈베리파이에서 쓰는 법**
```python
import torch
model = torch.hub.load('ultralytics/yolov5', 'custom', path='best_hornet_v1.pt')
model.conf = 0.5
df = model(frame).pandas().xyxy[0]          # frame: 카메라 이미지 (RGB)
is_hornet = (df['name'] == 'velutina').any()
```